# 🎯 Crawling Komentar YouTube — Analisis Sentimen Coretax
**Skripsi:** Analisis Sentimen Masyarakat terhadap Coretax pada Komentar YouTube  
**Menggunakan:** YouTube Data API v3  
**Output:** Dataset CSV siap labeling

---
## 📋 Cara Mendapatkan API Key
1. Buka https://console.cloud.google.com
2. Buat project baru → klik **'New Project'**
3. Di menu kiri → **'APIs & Services'** → **'Library'**
4. Cari **'YouTube Data API v3'** → klik **Enable**
5. Klik **'Create Credentials'** → pilih **'API Key'**
6. Copy API Key → paste di sel berikutnya

> ⚠️ **Kuota gratis:** 10.000 unit/hari. 1 request komentar = ~1-3 unit. Cukup untuk ~3.000-5.000 komentar per hari.

## ⚙️ LANGKAH 1 — Instalasi & Konfigurasi

In [ ]:
# Install library yang dibutuhkan
!pip install google-api-python-client pandas openpyxl tqdm --quiet
print('✅ Instalasi selesai!')

In [ ]:
# ============================================================
#  🔑 KONFIGURASI UTAMA — ISI BAGIAN INI SEBELUM MENJALANKAN
# ============================================================

API_KEY = 'ISI_API_KEY_KAMU_DI_SINI'  # Ganti dengan API Key dari Google Console

# Keyword pencarian video
KEYWORDS = [
    'coretax',
    'coretax DJP',
    'coretax pajak',
    'coretax error',
    'coretax bermasalah',
    'aplikasi coretax'
]

# Rentang waktu (format: YYYY-MM-DDThh:mm:ssZ)
DATE_START = '2024-12-01T00:00:00Z'
DATE_END   = '2025-12-31T23:59:59Z'

# Jumlah video per keyword (max 50)
MAX_VIDEOS_PER_KEYWORD = 20

# Jumlah komentar per video (max 500 per video)
MAX_COMMENTS_PER_VIDEO = 300

# Nama file output
OUTPUT_CSV   = 'dataset_coretax_youtube_raw.csv'
OUTPUT_EXCEL = 'dataset_coretax_youtube_labeling.xlsx'

print('✅ Konfigurasi berhasil dimuat!')
print(f'   Keyword  : {len(KEYWORDS)} keyword')
print(f'   Periode  : {DATE_START[:10]} s/d {DATE_END[:10]}')
print(f'   Max video: {MAX_VIDEOS_PER_KEYWORD} per keyword')
print(f'   Max komentar: {MAX_COMMENTS_PER_VIDEO} per video')

## 🔧 LANGKAH 2 — Inisialisasi API & Fungsi Utama

In [ ]:
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError
import pandas as pd
from tqdm import tqdm
import time
import re

# Inisialisasi YouTube API
youtube = build('youtube', 'v3', developerKey=API_KEY)
print('✅ YouTube API berhasil diinisialisasi!')

# ----------------------------------------------------------
# FUNGSI 1: Cari video berdasarkan keyword + rentang waktu
# ----------------------------------------------------------
def search_videos(keyword, max_results=20):
    videos = []
    try:
        request = youtube.search().list(
            part          = 'snippet',
            q             = keyword,
            type          = 'video',
            videoDuration = 'any',
            order         = 'relevance',
            publishedAfter= DATE_START,
            publishedBefore= DATE_END,
            relevanceLanguage = 'id',
            maxResults    = min(max_results, 50)
        )
        response = request.execute()

        for item in response.get('items', []):
            videos.append({
                'video_id'    : item['id']['videoId'],
                'judul_video' : item['snippet']['title'],
                'channel'     : item['snippet']['channelTitle'],
                'tanggal_upload': item['snippet']['publishedAt'][:10],
                'keyword_cari': keyword
            })
    except HttpError as e:
        print(f'⚠️  Error saat cari video "{keyword}": {e}')
    return videos


# ----------------------------------------------------------
# FUNGSI 2: Ambil detail video (jumlah view, like, komentar)
# ----------------------------------------------------------
def get_video_stats(video_id):
    try:
        response = youtube.videos().list(
            part = 'statistics',
            id   = video_id
        ).execute()
        stats = response['items'][0]['statistics']
        return {
            'view_count'   : int(stats.get('viewCount', 0)),
            'like_count'   : int(stats.get('likeCount', 0)),
            'comment_count': int(stats.get('commentCount', 0))
        }
    except:
        return {'view_count': 0, 'like_count': 0, 'comment_count': 0}


# ----------------------------------------------------------
# FUNGSI 3: Ambil komentar dari satu video
# ----------------------------------------------------------
def get_comments(video_id, video_info, max_comments=300):
    comments = []
    try:
        request = youtube.commentThreads().list(
            part       = 'snippet',
            videoId    = video_id,
            maxResults = 100,
            order      = 'relevance',
            textFormat = 'plainText'
        )

        while request and len(comments) < max_comments:
            response = request.execute()

            for item in response.get('items', []):
                snippet = item['snippet']['topLevelComment']['snippet']
                komentar = snippet['textDisplay'].strip()

                # Filter: minimal 5 kata, bukan hanya emoji
                if len(komentar.split()) < 5:
                    continue

                comments.append({
                    'video_id'      : video_id,
                    'judul_video'   : video_info['judul_video'],
                    'channel'       : video_info['channel'],
                    'tanggal_upload': video_info['tanggal_upload'],
                    'keyword_cari'  : video_info['keyword_cari'],
                    'view_count'    : video_info.get('view_count', 0),
                    'like_count'    : video_info.get('like_count', 0),
                    'penulis'       : snippet['authorDisplayName'],
                    'komentar'      : komentar,
                    'likes_komentar': snippet['likeCount'],
                    'tanggal_komentar': snippet['publishedAt'][:10],
                    'bulan_komentar': snippet['publishedAt'][:7],
                    'label'         : ''   # dikosongkan untuk labeling manual
                })

            # Ambil halaman berikutnya jika ada
            request = youtube.commentThreads().list_next(request, response)
            time.sleep(0.3)  # hindari rate limit

    except HttpError as e:
        err_str = str(e)
        if 'commentsDisabled' in err_str:
            print(f'   ⚠️  Komentar dinonaktifkan di video ini')
        elif 'quotaExceeded' in err_str:
            print('   🚨 KUOTA API HABIS! Hentikan dan coba besok.')
            raise
        else:
            print(f'   ⚠️  Error: {e}')

    return comments[:max_comments]


# ----------------------------------------------------------
# FUNGSI 4: Bersihkan teks komentar (untuk kolom preview)
# ----------------------------------------------------------
def clean_text(text):
    text = re.sub(r'http\S+', '', text)          # hapus URL
    text = re.sub(r'@\w+', '', text)             # hapus mention
    text = re.sub(r'#\w+', '', text)             # hapus hashtag
    text = re.sub(r'[^\w\s]', '', text)          # hapus tanda baca
    text = re.sub(r'\s+', ' ', text).strip()     # rapikan spasi
    return text.lower()

print('✅ Semua fungsi berhasil dimuat!')

## 🔍 LANGKAH 3 — Cari Video Coretax

In [ ]:
print('🔍 Mencari video YouTube tentang Coretax...\n')

all_videos = []
seen_ids   = set()

for keyword in tqdm(KEYWORDS, desc='Keyword'):
    videos = search_videos(keyword, MAX_VIDEOS_PER_KEYWORD)
    added  = 0
    for v in videos:
        if v['video_id'] not in seen_ids:
            seen_ids.add(v['video_id'])
            all_videos.append(v)
            added += 1
    print(f'  [{keyword}] → {added} video baru ditemukan')
    time.sleep(1)

print(f'\n✅ Total video unik ditemukan: {len(all_videos)}')

# Preview daftar video
df_videos = pd.DataFrame(all_videos)
print('\n📋 Preview daftar video:')
df_videos[['judul_video','channel','tanggal_upload','keyword_cari']].head(10)

In [ ]:
# Tampilkan semua video yang ditemukan
print(f'📹 Total {len(all_videos)} video akan di-crawl:\n')
for i, v in enumerate(all_videos, 1):
    print(f'{i:3}. [{v["tanggal_upload"]}] {v["channel"]:30s} | {v["judul_video"][:60]}')

## 📥 LANGKAH 4 — Crawling Komentar dari Semua Video

In [ ]:
print('📥 Mulai crawling komentar...\n')
print('='*60)

all_comments   = []
video_sukses   = 0
video_skip     = 0

for i, video in enumerate(all_videos, 1):
    vid_id = video['video_id']
    print(f'[{i:3}/{len(all_videos)}] {video["channel"]:25s} | {video["judul_video"][:45]}...')

    # Ambil statistik video
    stats = get_video_stats(vid_id)
    video.update(stats)

    # Skip video dengan 0 komentar
    if stats['comment_count'] == 0:
        print('   ⏭️  Tidak ada komentar, dilewati')
        video_skip += 1
        continue

    # Crawl komentar
    comments = get_comments(vid_id, video, MAX_COMMENTS_PER_VIDEO)

    if comments:
        all_comments.extend(comments)
        video_sukses += 1
        print(f'   ✅ {len(comments):4d} komentar | Total sejauh ini: {len(all_comments)}')
    else:
        video_skip += 1
        print('   ⚠️  Tidak ada komentar valid')

    time.sleep(0.5)  # jeda antar video

print('='*60)
print(f'\n🎉 Crawling selesai!')
print(f'   Video berhasil  : {video_sukses}')
print(f'   Video dilewati  : {video_skip}')
print(f'   Total komentar  : {len(all_comments)}')

## 🧹 LANGKAH 5 — Cleaning & Deduplikasi Dataset

In [ ]:
print('🧹 Membersihkan dataset...\n')

df = pd.DataFrame(all_comments)
print(f'Data mentah       : {len(df):,} baris')

# 1. Hapus duplikat berdasarkan isi komentar
df.drop_duplicates(subset='komentar', inplace=True)
print(f'Setelah dedup     : {len(df):,} baris')

# 2. Hapus komentar terlalu pendek (< 5 kata)
df = df[df['komentar'].str.split().str.len() >= 5]
print(f'Setelah filter panjang: {len(df):,} baris')

# 3. Tambahkan kolom komentar_bersih (untuk preprocessing NLP)
df['komentar_bersih'] = df['komentar'].apply(clean_text)

# 4. Hapus komentar_bersih yang kosong setelah cleaning
df = df[df['komentar_bersih'].str.len() > 0]
print(f'Setelah filter kosong : {len(df):,} baris')

# 5. Reset index
df.reset_index(drop=True, inplace=True)

# 6. Urutkan berdasarkan tanggal komentar
df['tanggal_komentar'] = pd.to_datetime(df['tanggal_komentar'])
df.sort_values('tanggal_komentar', inplace=True)
df.reset_index(drop=True, inplace=True)

print(f'\n✅ Dataset bersih: {len(df):,} komentar siap digunakan!')
print(f'   Kolom         : {list(df.columns)}')

## 📊 LANGKAH 6 — Statistik Dataset

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

print('='*50)
print('📊 STATISTIK DATASET CORETAX YOUTUBE')
print('='*50)
print(f'Total komentar valid : {len(df):,}')
print(f'Total video          : {df["video_id"].nunique()}')
print(f'Total channel        : {df["channel"].nunique()}')
print(f'Rentang tanggal      : {df["tanggal_komentar"].min().date()} s/d {df["tanggal_komentar"].max().date()}')
print(f'Rata-rata panjang    : {df["komentar"].str.split().str.len().mean():.1f} kata/komentar')

print('\n📋 Top 10 Channel Terbanyak:')
print(df['channel'].value_counts().head(10).to_string())

# ─── Grafik tren komentar per bulan ───
df['bulan'] = df['tanggal_komentar'].dt.to_period('M')
tren = df.groupby('bulan').size()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Plot 1: Tren per bulan
tren.plot(ax=axes[0], kind='bar', color='#1D9E75', edgecolor='white')
axes[0].set_title('Jumlah Komentar per Bulan', fontweight='bold')
axes[0].set_xlabel('Bulan')
axes[0].set_ylabel('Jumlah Komentar')
axes[0].tick_params(axis='x', rotation=45)

# Plot 2: Top channel
top_ch = df['channel'].value_counts().head(8)
top_ch.plot(ax=axes[1], kind='barh', color='#534AB7', edgecolor='white')
axes[1].set_title('Top Channel Sumber Komentar', fontweight='bold')
axes[1].set_xlabel('Jumlah Komentar')
axes[1].invert_yaxis()

plt.suptitle('Statistik Dataset Coretax YouTube', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('statistik_dataset.png', dpi=150, bbox_inches='tight')
plt.show()
print('\n✅ Grafik disimpan: statistik_dataset.png')

## 💾 LANGKAH 7 — Simpan Dataset

In [ ]:
# ─── Simpan CSV (dataset lengkap) ───
df.to_csv(OUTPUT_CSV, index=False, encoding='utf-8-sig')
print(f'✅ Dataset lengkap disimpan: {OUTPUT_CSV}')

# ─── Simpan Excel (untuk labeling manual) ───
# Kolom yang ditampilkan untuk labeling
kolom_labeling = [
    'tanggal_komentar',
    'bulan_komentar',
    'channel',
    'judul_video',
    'penulis',
    'komentar',
    'komentar_bersih',
    'likes_komentar',
    'label'   # ← kolom kosong untuk diisi manual: positif/negatif/netral
]

df_label = df[kolom_labeling].copy()

with pd.ExcelWriter(OUTPUT_EXCEL, engine='openpyxl') as writer:
    df_label.to_excel(writer, index=True, sheet_name='Data Labeling')

    # Sheet 2: Statistik ringkas
    stats = pd.DataFrame({
        'Keterangan': [
            'Total Komentar',
            'Total Video',
            'Total Channel',
            'Tanggal Mulai',
            'Tanggal Akhir',
            'Rata-rata Panjang Komentar'
        ],
        'Nilai': [
            len(df),
            df['video_id'].nunique(),
            df['channel'].nunique(),
            str(df['tanggal_komentar'].min().date()),
            str(df['tanggal_komentar'].max().date()),
            f"{df['komentar'].str.split().str.len().mean():.1f} kata"
        ]
    })
    stats.to_excel(writer, index=False, sheet_name='Statistik')

print(f'✅ File Excel labeling disimpan: {OUTPUT_EXCEL}')
print(f'\n📌 INSTRUKSI LABELING:')
print(f'   Buka file Excel → Sheet "Data Labeling"')
print(f'   Isi kolom "label" dengan salah satu:')
print(f'   - positif  → komentar mendukung/puas dengan Coretax')
print(f'   - negatif  → komentar mengkritik/kecewa dengan Coretax')
print(f'   - netral   → komentar informatif/tidak memihak')

## 📥 LANGKAH 8 — Download File ke Komputer

In [ ]:
from google.colab import files

print('📥 Mengunduh file dataset...')
files.download(OUTPUT_CSV)
files.download(OUTPUT_EXCEL)
files.download('statistik_dataset.png')
print('\n✅ Semua file berhasil diunduh!')
print('   File yang diunduh:')
print(f'   1. {OUTPUT_CSV}         ← dataset lengkap')
print(f'   2. {OUTPUT_EXCEL}  ← untuk labeling manual di Excel')
print(f'   3. statistik_dataset.png ← grafik untuk BAB 4')

## 🔁 BONUS — Crawling dari Video ID Spesifik

Jika kamu sudah tahu video mana yang ingin diambil (misal dari channel berita besar seperti Kompas TV, CNN Indonesia, Najwa Shihab, dll), gunakan sel ini untuk langsung ambil komentar tanpa proses pencarian.

In [ ]:
# ─── Crawling dari Video ID Tertentu ───
# Ambil video_id dari URL YouTube: https://www.youtube.com/watch?v=VIDEO_ID_DI_SINI

VIDEO_IDS_MANUAL = [
    # Contoh: video dari Kompas TV, CNN Indonesia, CNBC Indonesia, dll
    # 'xxxxxxxxxxx',
    # 'yyyyyyyyyyy',
    # Isi dengan video_id yang relevan
]

if VIDEO_IDS_MANUAL:
    extra_comments = []
    for vid_id in tqdm(VIDEO_IDS_MANUAL, desc='Video Manual'):
        stats = get_video_stats(vid_id)
        info  = {
            'video_id'    : vid_id,
            'judul_video' : 'Manual Input',
            'channel'     : 'Manual Input',
            'tanggal_upload': '-',
            'keyword_cari': 'manual',
            **stats
        }
        comments = get_comments(vid_id, info, MAX_COMMENTS_PER_VIDEO)
        extra_comments.extend(comments)
        print(f'  ✅ {vid_id}: {len(comments)} komentar')
        time.sleep(0.5)

    df_extra = pd.DataFrame(extra_comments)
    df_extra.drop_duplicates(subset='komentar', inplace=True)
    print(f'\n✅ Total komentar manual: {len(df_extra)}')
    df_extra.to_csv('dataset_coretax_manual.csv', index=False, encoding='utf-8-sig')
    files.download('dataset_coretax_manual.csv')
else:
    print('ℹ️  Tidak ada video_id manual. Isi VIDEO_IDS_MANUAL di atas untuk menggunakannya.')

## ✅ Rangkuman Alur Kerja

```
1. Isi API Key & konfigurasi
2. Jalankan Langkah 1–8 secara berurutan
3. Download file Excel (dataset_coretax_youtube_labeling.xlsx)
4. Labeling manual kolom 'label' di Excel
   → positif / negatif / netral
5. Upload kembali ke Colab untuk preprocessing & modeling
```

## ⚠️ Tips Penting

| Masalah | Solusi |
|---|---|
| Kuota API habis (10.000 unit/hari) | Jalankan besok, data tersimpan otomatis |
| Komentar dinonaktifkan | Video tersebut otomatis dilewati |
| Data sedikit (<500) | Tambah keyword atau MAX_VIDEOS_PER_KEYWORD |
| Error 403 | Periksa API Key sudah benar & YouTube API aktif |
| Session Colab timeout | Jalankan ulang dari Langkah 4 saja |
